In [1]:
from __future__ import annotations

import os

from abc import ABC, abstractmethod
from collections import defaultdict
import dataclasses
import functools
from functools import cached_property
import heapq
import json
from pathlib import Path
from typing import Any, Iterable, Literal

import numpy as np
import polars as pl
import scipy.sparse as sp
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader

from datasets import Dataset, DatasetDict, load_dataset


In [3]:
from sansa import SANSA,  ICFGramianFactorizerConfig, SANSAConfig, UMRUnitLowerTriangleInverterConfig

In [6]:
import torch

# Utils & Helpers

In [13]:
class Constants:
    HOUR_SECONDS = 60 * 60
    DAY_SECONDS = 24 * HOUR_SECONDS

    GAP_SIZE = HOUR_SECONDS // 2
    VAL_SIZE = 1 * DAY_SECONDS
    TEST_SIZE = 1 * DAY_SECONDS

    LAST_TIMESTAMP = 26000000
    TEST_TIMESTAMP = LAST_TIMESTAMP - TEST_SIZE

    TRACK_LISTEN_THRESHOLD = 50

    NUM_RANKED_ITEMS = 100

    METRICS = [
        "ndcg@10",
        "ndcg@100",
        "recall@10",
        "recall@100",
        "coverage@10",
        "coverage@100",
    ]

    IDEAL_METRICS = {
        'most_pop': {
            "ndcg@10": 0.0046,
            "ndcg@100": 0.0097,
            "recall@10": 0.0083,
            "recall@100": 0.0222,
            "coverage@10": 0.0001,
            "coverage@100": 0.0006,
        },
        'decay_pop': {
            "ndcg@10": 0.0180,
            "ndcg@100": 0.0269,
            "recall@10": 0.0333,
            "recall@100": 0.0651,
            "coverage@10": 0.0001,
            "coverage@100": 0.0006,
        },
        'item_knn': {
            "ndcg@10": 0.0125,
            "ndcg@100": 0.0251,
            "recall@10": 0.0199,
            "recall@100": 0.0648,
            "coverage@10": 0.1050,
            "coverage@100": 0.3922,
        },
        'sansa': {
            "ndcg@10": 0.0068,
            "ndcg@100": 0.0203,
            "recall@10": 0.0105,
            "recall@100": 0.0616,
            "coverage@10": 0.0194,
            "coverage@100": 0.1601,
        }
    }
     

In [7]:
def rank_items(users: Embeddings, items: Embeddings, num_items: int, batch_size: int = 128) -> Ranked:
    assert users.device == items.device

    num_users = users.ids.shape[0]

    scores = users.embeddings.new_empty((num_users, num_items))
    item_ids = users.embeddings.new_empty((num_users, num_items), dtype=torch.long)

    for batch_idx in tqdm(range((num_users + batch_size - 1) // batch_size), desc="Calc topk by batches"):
        start_idx = batch_idx * batch_size
        end_idx = (batch_idx + 1) * batch_size

        batch_scores = users.embeddings[start_idx:end_idx, :] @ items.embeddings.T

        sort_indices = batch_scores.topk(num_items, dim=-1).indices
        scores[start_idx:end_idx, :] = torch.gather(batch_scores, dim=-1, index=sort_indices)

        item_ids[start_idx:end_idx, :] = torch.gather(
            items.ids.expand(sort_indices.shape[0], items.ids.shape[0]), dim=-1, index=sort_indices
        )

    return Ranked(user_ids=users.ids, item_ids=item_ids, scores=scores, num_item_ids=items.ids.shape[0])


@dataclasses.dataclass
class Ranked:
    user_ids: torch.Tensor
    item_ids: torch.Tensor
    scores: torch.Tensor | None = None
    num_item_ids: int | None = None  # number of all items. Useful for coverage and etc.

    def __post_init__(self):
        if self.scores is None:
            self.scores = torch.arange(
                self.item_ids.shape[1], 0, -1, device=self.item_ids.device, dtype=torch.float32
            ).expand((self.user_ids.shape[0], self.item_ids.shape[1]))

        assert self.user_ids.dim() == 1
        assert self.scores.dim() == 2
        assert self.scores.shape == self.item_ids.shape
        assert self.user_ids.shape[0] == self.scores.shape[0]

        assert self.user_ids.device == self.scores.device == self.item_ids.device

        assert torch.all(self.scores[:, :-1] >= self.scores[:, 1:]), "scores should be sorted"

        if not torch.all(self.user_ids[:-1] <= self.user_ids[1:]):
            indexes = torch.argsort(self.user_ids, descending=False)
            self.item_ids = self.item_ids[indexes, :]
            self.scores = self.scores[indexes, :]
            self.user_ids = self.user_ids[indexes]

    @property
    def device(self):
        return self.user_ids.device


@dataclasses.dataclass
class Targets:
    user_ids: torch.Tensor
    item_ids: list[torch.Tensor]

    def __post_init__(self):
        assert len(self.item_ids) > 0
        assert self.user_ids.dim() == 1
        assert self.user_ids.shape[0] == len(self.item_ids)
        assert all(x.dim() == 1 for x in self.item_ids), "all ids should be 1D"

        assert all(x.device == self.item_ids[0].device for x in self.item_ids), "all ids should be on the same device"
        assert self.user_ids.device == self.item_ids[0].device

        if not torch.all(self.user_ids[:-1] <= self.user_ids[1:]):
            indexes = torch.argsort(self.user_ids, descending=False)
            self.item_ids = [self.item_ids[i] for i in indexes]
            self.user_ids = self.user_ids[indexes]

        assert torch.all(self.user_ids[:-1] < self.user_ids[1:]), "user_ids should be unique"

    @cached_property
    def lengths(self):
        return torch.tensor([ids.shape[0] for ids in self.item_ids], device=self.item_ids[0].device)

    def __len__(self):
        return len(self.item_ids)

    @property
    def device(self):
        return self.user_ids.device

    @classmethod
    def from_sequential(cls, df: pl.LazyFrame | pl.DataFrame, device: torch.device | str) -> 'Targets':
        return cls(
            df.select("uid")["uid"].to_torch().to(device),
            [torch.tensor(x, device=device) for x in df.select("item_id")["item_id"].to_list()],
        )



@dataclasses.dataclass
class Embeddings:
    ids: torch.Tensor
    embeddings: torch.Tensor

    def __post_init__(self):
        assert self.ids.dim() == 1
        assert self.embeddings.dim() == 2
        assert self.ids.shape[0] == self.embeddings.shape[0]

        assert self.ids.device == self.embeddings.device

        if not torch.all(self.ids[:-1] <= self.ids[1:]):
            indexes = torch.argsort(self.ids, descending=False)
            self.embeddings = self.embeddings[indexes, :]
            self.ids = self.ids[indexes]

        assert torch.all(self.ids[:-1] < self.ids[1:]), "ids should be unique"

    @property
    def device(self):
        return self.ids.device

    def save(self, file_path: str):
        ids_np = self.ids.cpu().numpy()
        embeddings_np = self.embeddings.cpu().numpy()
        np.savez(file_path, ids=ids_np, embeddings=embeddings_np)

    @classmethod
    def load(cls, file_path: str, device: torch.device = torch.device('cpu')) -> 'Embeddings':
        with np.load(file_path) as data:
            ids_np = data['ids']
            embeddings_np = data['embeddings']

        ids = torch.from_numpy(ids_np).to(device)
        embeddings = torch.from_numpy(embeddings_np).to(device)

        return cls(ids=ids, embeddings=embeddings)


# Metrics 

In [8]:
def cut_off_ranked(ranked: Ranked, targets: Targets) -> Ranked:
    mask = torch.isin(ranked.user_ids, targets.user_ids)

    assert ranked.scores is not None

    ranked = Ranked(
        user_ids=ranked.user_ids[mask],
        scores=ranked.scores[mask, :],
        item_ids=ranked.item_ids[mask, :],
        num_item_ids=ranked.num_item_ids,
    )

    assert ranked.item_ids.shape[0] == len(targets), "Ranked doesn't contain all targets.user_ids"

    return ranked


class Metric(ABC):
    @abstractmethod
    def __call__(
        self, ranked: Ranked | None, targets: Targets | None, target_mask: torch.Tensor | None, ks: Iterable[int]
    ) -> dict[int, float]:
        pass


class Recall(Metric):
    def __call__(
        self, ranked: Ranked | None, targets: Targets, target_mask: torch.Tensor, ks: Iterable[int]
    ) -> dict[int, float]:
        assert all(0 < k <= target_mask.shape[1] for k in ks)

        values = {}

        for k in ks:
            num_positives = targets.lengths.to(torch.float32)
            num_positives[num_positives == 0] = torch.inf

            values[k] = target_mask[:, :k].to(torch.float32).sum(dim=-1) / num_positives

            values[k] = torch.mean(values[k]).item()

        return values


class DCG(Metric):
    def __call__(
        self, ranked: Ranked | None, targets: Targets | None, target_mask: torch.Tensor, ks: Iterable[int]
    ) -> dict[int, float]:
        assert all(0 < k <= target_mask.shape[1] for k in ks)

        values = {}

        discounts = 1.0 / torch.log2(
            torch.arange(2, target_mask.shape[1] + 2, device=target_mask.device, dtype=torch.float32)
        )

        for k in ks:
            dcg_k = torch.sum(target_mask[:, :k] * discounts[:k], dim=1)
            values[k] = torch.mean(dcg_k).item()

        return values


class NDCG(Metric):
    def __call__(
        self, ranked: Ranked | None, targets: Targets, target_mask: torch.Tensor, ks: Iterable[int]
    ) -> dict[int, float]:
        actual_dcg = DCG()(ranked, targets, target_mask, ks)

        ideal_target_mask = (
            torch.arange(target_mask.shape[1], device=targets.device)[None, :] < targets.lengths[:, None]
        ).to(torch.float32)
        assert target_mask.shape == ideal_target_mask.shape

        ideal_dcg = DCG()(ranked, targets, ideal_target_mask, ks)

        ndcg_values = {k: (actual_dcg[k] / ideal_dcg[k] if ideal_dcg[k] != 0 else 0.0) for k in ks}

        return ndcg_values


class Coverage(Metric):
    def __init__(self, cut_off_ranked: bool = False):
        self.cut_off_ranked = cut_off_ranked

    def __call__(
        self, ranked: Ranked, targets: Targets | None, target_mask: torch.Tensor | None, ks: Iterable[int]
    ) -> dict[int, float]:
        if self.cut_off_ranked:
            assert targets is not None
            ranked = cut_off_ranked(ranked, targets)

        assert all(0 < k <= ranked.item_ids.shape[1] for k in ks)

        assert ranked.num_item_ids is not None

        values = {}
        for k in ks:
            values[k] = ranked.item_ids[:, :k].flatten().unique().shape[0] / ranked.num_item_ids

        return values

# Metrics Utilities

In [9]:
REGISTERED_METRIC_FN = {
    "recall": Recall(),
    "ndcg": NDCG(),
    "coverage": Coverage(cut_off_ranked=False),
}


def calc_metrics(ranked: Ranked, targets: Targets, metrics: list[str]) -> dict[str, Any]:
    grouped_metrics = _parse_metrics(metrics)

    result = {}

    target_mask = create_target_mask(ranked, targets)

    for name, ks in grouped_metrics.items():
        result[name] = REGISTERED_METRIC_FN[name](ranked, targets, target_mask, ks=ks)

    return result


def _parse_metrics(metric_names: list[str]) -> dict[str, list[int]]:
    parsed_metrics = []

    for metric in metric_names:
        parts = metric.split('@')
        name = parts[0]

        assert len(parts) > 1, f"Invalid metric: {metric}, specify @k"

        value = int(parts[1])
        parsed_metrics.append((name, value))

    metrics = defaultdict(list)
    for m in parsed_metrics:
        metrics[m[0]].append(m[1])

    return metrics


def create_target_mask(ranked: Ranked, targets: Targets) -> torch.Tensor:
    ranked = cut_off_ranked(ranked, targets)

    assert ranked.device == targets.device
    assert ranked.item_ids.shape[0] == len(targets)

    target_mask = ranked.item_ids.new_zeros(ranked.item_ids.shape, dtype=torch.float32)

    for i, target in enumerate(tqdm(targets.item_ids, desc="Making target mask")):
        target_mask[i, torch.isin(ranked.item_ids[i], target)] = 1.0

    return target_mask

# Dataset

In [10]:
class YambdaDataset:
    INTERACTIONS = frozenset([
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ])

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(self, event_type: Literal[
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ]) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]

     

In [11]:
# Load data using YambdaDataset
yambda = YambdaDataset(dataset_type="flat", dataset_size="50m")
likes_ds = yambda.interaction("likes")  # load all likes
print(f"Loaded {len(likes_ds):,} records (likes)")

likes_df = likes_ds.to_polars()
likes_df.head()
     

INFO:httpx2:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://huggingface.co/datasets/yandex/yambda/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/yandex/yambda/dd6f3a19eef5866e346c3270e098baa641a44948/README.md?%2Fdatasets%2Fyandex%2Fyambda%2Fresolve%2Fmain%2FREADME.md=&etag=%221ee8199864f4748462a26eb9e15745af7481e6bc%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://huggingface.co/datasets/yandex/yambda/resolve/dd6f3a19eef5866e346c3270e098baa641a44948/yambda.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/yandex/yambda/yandex/yambda.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://huggingface.co/api/datasets/yandex/yambda/revision/dd6f3a19eef5866e346c3270e098baa641a44948 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https

Loaded 881,456 records (likes)


uid,timestamp,item_id,is_organic
u32,u32,u32,u8
100,44755,732449,1
100,1155860,6568592,0
100,1259125,5411243,1
100,1260005,7371186,0
100,1263935,4943655,0


In [14]:
def flat_split_train_val_test(
    df: pl.LazyFrame,
    test_timestamp: int,
    val_size: int = 0,
    gap_size: int = Constants.GAP_SIZE,
    drop_non_train_items: bool = False,
    engine: str = "streaming",
) -> tuple[pl.LazyFrame, pl.LazyFrame | None, pl.LazyFrame]:

    # Filter out new items
    def drop(df: pl.LazyFrame, unique_train_item_ids) -> pl.LazyFrame:
        if not drop_non_train_items:
            return df

        return (
            df.with_columns(
                pl.col("item_id").is_in(unique_train_item_ids.get_column("item_id").implode()).alias("item_id_in_train")
            )
            .filter("item_id_in_train")
            .drop("item_id_in_train")
        )

    train_timestamp = test_timestamp - gap_size - val_size - (gap_size if val_size != 0 else 0)

    assert gap_size >= 0
    assert val_size >= 0
    assert train_timestamp > 0

    df_lazy = df.lazy()

    # Create train subset
    train = df_lazy.filter(pl.col("timestamp") < train_timestamp)

    # Warm-start setup
    unique_train_uids = train.select("uid").unique().collect(engine=engine)
    unique_train_item_ids = train.select("item_id").unique().collect(engine=engine)

    # Create validation subset
    validation = None
    if val_size != 0:
        validation = (
            df_lazy.filter(
                (pl.col("timestamp") >= test_timestamp - val_size - gap_size)
                & (pl.col("timestamp") < test_timestamp - gap_size)
            )
            #
            .with_columns(
                pl.col("uid").is_in(unique_train_uids.get_column("uid").implode()).alias("uid_in_train")
            )  # to prevent filter reordering
            .filter("uid_in_train")
            .drop("uid_in_train")
        )
        validation = drop(validation, unique_train_item_ids)

    # Create evaluation subset
    test = (
        df_lazy.filter(pl.col("timestamp") >= test_timestamp)
        #
        .with_columns(
            pl.col("uid").is_in(unique_train_uids.get_column("uid").implode()).alias("uid_in_train")
        )  # to prevent filter reordering
        .filter("uid_in_train")
        .drop("uid_in_train")
    )
    test = drop(test, unique_train_item_ids)

    return train, validation, test
     

In [15]:
train_likes_df, _, test_likes_df = flat_split_train_val_test(
    df=likes_df,
    test_timestamp=Constants.TEST_TIMESTAMP,
    val_size=0,  # No validation is needed
    gap_size=Constants.GAP_SIZE
)

train_likes_df = train_likes_df.collect()
test_likes_df = test_likes_df.collect()

In [16]:
train_likes_df.head()

uid,timestamp,item_id,is_organic
u32,u32,u32,u8
100,44755,732449,1
100,1155860,6568592,0
100,1259125,5411243,1
100,1260005,7371186,0
100,1263935,4943655,0


In [17]:

test_likes_df.head()

uid,timestamp,item_id,is_organic
u32,u32,u32,u8
700,25939545,3383415,0
1400,25993865,7707460,1
1400,25993945,6005845,1
1400,25993945,7539311,0
1400,25993975,8908728,1
